Notebook ini melakukan **prediksi sentimen** ulasan aplikasi OVO menggunakan:
- `model.pkl` — model klasifikasi sentimen
- `bow_vectorizer.pkl` — vektorizer Bag-of-Words
- `label_encoder.pkl` — encoder label sentimen
- `metadata.json` — metadata model

> Pipeline preprocessing diimport dari `datatext_preprocessing.py`

# Install & Import Library

In [12]:
# Install dependensi yang diperlukan
!pip install emoji nltk PySastrawi swifter -q

import pickle, json, os, importlib.util, sys
import numpy as np
import pandas as pd
import warnings
from joblib import load as joblib_load
warnings.filterwarnings('ignore')

print('Library berhasil diimport')

Library berhasil diimport


# Load Modul Preprocessing

In [13]:
# Pastikan datatext_preprocessing.py ada di /content/
_PREPROC_PATH = '/content/datatext_preprocessing.py'

spec = importlib.util.spec_from_file_location('datatext_preprocessing', _PREPROC_PATH)
preproc_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preproc_mod)

Total entri kamus slang: 839


In [14]:
# Ambil fungsi & variabel yang dibutuhkan
preprocess_for_sentiment = preproc_mod.preprocess_for_sentiment
slang_dict               = preproc_mod.slang_dict
NEGATION_WORDS           = preproc_mod.NEGATION_WORDS
SAFE_STOPWORDS           = preproc_mod.SAFE_STOPWORDS

print(' Modul preprocessing berhasil dimuat')
print(f'   Total entri kamus slang : {len(slang_dict)}')
print(f'   Total kata negasi       : {len(NEGATION_WORDS)}')

 Modul preprocessing berhasil dimuat
   Total entri kamus slang : 839
   Total kata negasi       : 33


# Load Model

In [16]:
MODEL_DIR = '/content'

def load_artifact(filename: str):
    path = os.path.join(MODEL_DIR, filename)

    print(f"\nLoading {filename}...")

    # Coba pakai joblib dulu (paling sering jadi penyebab)
    try:
        obj = joblib_load(path)
        print(f'    {filename} berhasil dimuat (joblib)')
        return obj
    except Exception as e:
        print(f'    joblib gagal: {e}')

    # Fallback ke pickle
    try:
        with open(path, 'rb') as f:
            obj = pickle.load(f)
        print(f'    {filename} berhasil dimuat (pickle)')
        return obj
    except Exception as e:
        print(f'    pickle juga gagal: {e}')
        raise e

In [17]:
print('=' * 60)
print('MEMUAT ARTEFAK MODEL')
print('=' * 60)

model          = load_artifact('model.pkl')
bow_vectorizer = load_artifact('bow_vectorizer.pkl')
label_encoder  = load_artifact('label_encoder.pkl')

# Metadata
with open(os.path.join(MODEL_DIR, 'metadata.json'), 'r') as f:
    metadata = json.load(f)

print(f'\n Metadata Model:')
for k, v in metadata.items():
    print(f'   {k}: {v}')

MEMUAT ARTEFAK MODEL

Loading model.pkl...
    model.pkl berhasil dimuat (joblib)

Loading bow_vectorizer.pkl...
    bow_vectorizer.pkl berhasil dimuat (joblib)

Loading label_encoder.pkl...
    label_encoder.pkl berhasil dimuat (joblib)

 Metadata Model:
   model_name: MLP
   feature_extraction: BoW (CountVectorizer)
   saved_at: 2026-04-21 02:42:46
   classes: ['Negatif', 'Netral', 'Positif']
   optimal_thresholds: {'Negatif': 0.23, 'Netral': 0.05, 'Positif': 0.55}
   requires_scaler: False
   bow_params: {'max_features': 5000, 'ngram_range': [1, 2], 'binary': False, 'min_df': 2, 'max_df': 0.95}
   metrics: {'accuracy': 0.8745, 'precision': 0.8546, 'recall': 0.8745, 'f1_score': 0.8552}


# Fungsi Inferensi

In [18]:
def predict_sentiment(texts, model, vectorizer, label_encoder, slang_dict, verbose=True):
    """
    Melakukan prediksi sentimen pada daftar teks ulasan.

    Returns
    -------
    pd.DataFrame dengan kolom:
        review_original, text_cleaned, tokens, tokens_with_negation,
        predicted_label, predicted_encoded, confidence, confidence_all
    """
    results = []

    for i, raw_text in enumerate(texts):
        if verbose:
            preview = raw_text[:60] + '...' if len(raw_text) > 60 else raw_text
            print(f'  [{i+1}/{len(texts)}] Memproses: "{preview}"')

        # 1. Preprocessing
        text_cleaned, tokens, tokens_with_negation, tokens_stemmed = \
            preprocess_for_sentiment(
                raw_text, slang_dict,
                use_emoji=True,
                use_stemming=False,
                use_negation_marking=True,
                keep_negations=True
            )

        # 2. Vektorisasi
        text_for_vec = ' '.join(tokens_with_negation)
        X = vectorizer.transform([text_for_vec])

        # 3. Prediksi
        pred_encoded = model.predict(X)[0]
        pred_label   = label_encoder.inverse_transform([pred_encoded])[0]

        # 4. Confidence
        confidence, confidence_all = None, None
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(X)[0]
            confidence = float(np.max(proba))
            confidence_all = {
                label_encoder.inverse_transform([j])[0]: round(float(p), 4)
                for j, p in enumerate(proba)
            }

        results.append({
            'review_original'      : raw_text,
            'text_cleaned'         : text_cleaned,
            'tokens'               : tokens,
            'tokens_with_negation' : tokens_with_negation,
            'predicted_label'      : pred_label,
            'predicted_encoded'    : pred_encoded,
            'confidence'           : confidence,
            'confidence_all'       : confidence_all,
        })

    return pd.DataFrame(results)

print('Fungsi predict_sentiment siap digunakan')

Fungsi predict_sentiment siap digunakan


# Data review di app store

Berikut adalah data review aplikasi OVO yang saya dapatkan di app store.

In [23]:
dummy_reviews = [
    # Bintang 5
    "Baru kali ini dapet CS respond cepet, nggak template dan langsung to the point ke permasalahan.",

    # Bintang 5
    "Bagus suka banget sama aplikasinya.",

    # Bintang 5
    "Saya sangat bangga dan sangat tertarik karena OVO adalah aplikasi transaksi yang sangat terpercaya.",

    # Bintang 1
    "Sore tanggal 20 Juni saya top up sebesar 200 ribu, tapi saat cek saldo di aplikasi masih Rp 0. "
    "Di history hanya ada transaksi top up saja. Saya tidak bisa transaksi karena dianggap saldo kosong. "
    "Bagaimana ini?",

    # Bintang 1
    "Saya melakukan pembayaran di Din Tai Fung Gandaria City dengan split payment OVO Cash dan point. "
    "Ternyata OVO sedang error. OVO cash saya dikembalikan tapi point saya tidak. "
    "Sudah kontak call center tidak ada respon. Saya sangat kecewa, tolong kembalikan point saya.",

    # Bintang 1
    "App tidak berguna, sign in saja gagal terus apalagi mau transaksi. "
    "Pantas saja ratingnya jelek.",

    # Bintang 1
    "Tidak profesional, pelayanan sangat mengecewakan.",

    # Bintang 1
    "Sangat kecewa dan merasa tertipu menggunakan OVO. Pembayaran hanya bisa menggunakan OVO cash, "
    "deal sering gagal saat payment, CS sulit dihubungi, dan refund sangat ribet. "
    "Untuk upgrade harus ke kantor cabang dan tidak bisa online. Sangat merepotkan dan mengecewakan."
]

In [20]:
for i, r in enumerate(dummy_reviews, 1):
    preview = r[:110] + '...' if len(r) > 110 else r
    print(f'\n[{i}] {preview}')


[1] Baru kali ini dapet CS respond cepet, nggak template dan langsung to the point ke permasalahan.

[2] Bagusss i likee, sukaaaa banget sama aplikasinya.

[3] Saya sangat bangga dan sangat tertarik karena OVO adalah aplikasi transaksi yang sangat terpercaya.

[4] Sore tanggal 20 Juni saya top up sebesar 200 ribu, tapi saat cek saldo di aplikasi masih Rp 0. Di history hany...

[5] Saya melakukan pembayaran di Din Tai Fung Gandaria City dengan split payment OVO Cash dan point. Ternyata OVO ...

[6] App tidak berguna, sign in saja gagal terus apalagi mau transaksi. Pantas saja ratingnya jelek.

[7] Tidak profesional, pelayanan sangat mengecewakan.

[8] Sangat kecewa dan merasa tertipu menggunakan OVO. Pembayaran hanya bisa menggunakan OVO cash, deal sering gaga...


# Inferensi

In [24]:
print('\n' + '='*60)
print('MENJALANKAN INFERENCE')
print('='*60 + '\n')

df_pred = predict_sentiment(
    texts         = dummy_reviews,
    model         = model,
    vectorizer    = bow_vectorizer,
    label_encoder = label_encoder,
    slang_dict    = slang_dict,
    verbose       = True
)

print('\n Inference selesai!')


MENJALANKAN INFERENCE

  [1/8] Memproses: "Baru kali ini dapet CS respond cepet, nggak template dan lan..."
  [2/8] Memproses: "Bagus suka banget sama aplikasinya."
  [3/8] Memproses: "Saya sangat bangga dan sangat tertarik karena OVO adalah apl..."
  [4/8] Memproses: "Sore tanggal 20 Juni saya top up sebesar 200 ribu, tapi saat..."
  [5/8] Memproses: "Saya melakukan pembayaran di Din Tai Fung Gandaria City deng..."
  [6/8] Memproses: "App tidak berguna, sign in saja gagal terus apalagi mau tran..."
  [7/8] Memproses: "Tidak profesional, pelayanan sangat mengecewakan."
  [8/8] Memproses: "Sangat kecewa dan merasa tertipu menggunakan OVO. Pembayaran..."

 Inference selesai!


In [27]:
LABEL_EMOJI = {
    'positif'  : '🟢 POSITIF',
    'negatif'  : '🔴 NEGATIF',
    'netral'   : '🟡 NETRAL',
    'positive' : '🟢 POSITIVE',
    'negative' : '🔴 NEGATIVE',
    'neutral'  : '🟡 NEUTRAL',
    '1'        : '🟢 POSITIF',
    '-1'       : '🔴 NEGATIF',
    '0'        : '🟡 NETRAL',
}

In [26]:
print('\n' + '='*65)
print('HASIL PREDIKSI SENTIMEN')
print('='*65)

for i, row in df_pred.iterrows():
    label_display = LABEL_EMOJI.get(str(row['predicted_label']).lower(),
                                    f"⚪ {row['predicted_label']}")
    conf_str = f"{row['confidence']*100:.1f}%" if row['confidence'] is not None else 'N/A'

    print(f"\n{'─'*65}")
    print(f"📝 Review #{i+1}")
    orig = row['review_original']
    print(f"   Original   : {orig[:90]}{'...' if len(orig) > 90 else ''}")
    cln = row['text_cleaned']
    print(f"   Cleaned    : {cln[:90]}{'...' if len(cln) > 90 else ''}")
    print(f"   Tokens     : {row['tokens_with_negation'][:8]}")
    print(f"   ➜ Prediksi : {label_display}")
    print(f"   Confidence : {conf_str}")
    if row['confidence_all'] is not None:
        dist = '  |  '.join([f"{k}: {v*100:.1f}%" for k, v in row['confidence_all'].items()])
        print(f"   Distribusi : {dist}")

print(f"\n{'─'*65}")


HASIL PREDIKSI SENTIMEN

─────────────────────────────────────────────────────────────────
📝 Review #1
   Original   : Baru kali ini dapet CS respond cepet, nggak template dan langsung to the point ke permasal...
   Cleaned    : kali customer service respond cepat tidak template langsung the point permasalahan
   Tokens     : ['kali', 'customer', 'service', 'respond', 'cepat', 'NOT_template', 'langsung', 'the']
   ➜ Prediksi : 🟢 POSITIF
   Confidence : 90.4%
   Distribusi : Negatif: 8.7%  |  Netral: 0.9%  |  Positif: 90.4%

─────────────────────────────────────────────────────────────────
📝 Review #2
   Original   : Bagus suka banget sama aplikasinya.
   Cleaned    : bagus suka banget aplikasinya
   Tokens     : ['bagus', 'suka', 'banget', 'aplikasinya']
   ➜ Prediksi : 🟢 POSITIF
   Confidence : 98.8%
   Distribusi : Negatif: 1.1%  |  Netral: 0.1%  |  Positif: 98.8%

─────────────────────────────────────────────────────────────────
📝 Review #3
   Original   : Saya sangat bangga dan sa